In [1]:
# THIS CODE USES CUPY; FOR MORE THAN 500 GRIDPOINTS
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from PIL import Image
# from numba import jit, prange
import time 

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import os
from scipy.signal import find_peaks
from scipy.integrate import quad
from scipy.optimize import root_scalar
from matplotlib.animation import PillowWriter, FuncAnimation
import ipywidgets as widgets
from IPython.display import display
import zarr
import json

print("Fin")

/home/nehadesigar/pixi_env/.pixi/envs/default/lib/python3.12/site-packages/cupy/_environment.py:670: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


Fin


In [2]:
# Independent parameters (free to edit)

Na = 0.5 # Units: M 
T = 303.15 # Units: K
valence = 4
duration = 250 * 10**5 # In timesteps of dt
gridpoints = 128 # Number of points
dx = 10 # Units: nm
dt = 1.E-5 # Units: sec
rho_mean = 3E-5 # Initial mean density of nanostar A, found by spinodal (rho dense + rho dilute)/2 for the value of T used
save_interval = 10**5

grid_length = dx * gridpoints # Total length (nm)
inv_dx2 = 1.0 / (dx * dx)

# Establishes constants
M = 1 # Units: (nm s)^-1
vb = 1.66 # Units: nm^3
kB = 1.314E-23 * 0.24 # Units: cal/K (1J=0.24cal)
mol = 6.02E23
dHa = -42000 # Units: cal/mol 
dS1 = 1.84 * cp.log(Na) # Units: cal/mol K
dS0 = -120 # Units: cal/mol K at 1M NaCl
floor = 1E-12 # Minimum value for arrays
num_saves = duration // save_interval + 1 # Number of saved values

Da = vb * cp.exp(-(dHa - T * (dS0 + dS1)) / (mol * kB * T))
Db = Da

length_a = 16
length_b = 20

B2aa = 1121 # Units: nm^3
B2ab = 1467 # Units: nm^3
B2bb = 1878 # Units: nm^3

Ka = 1.0E6 # Units: nm^5 
Kb = 1.0E6 # Units: nm^5 

print("Fin")

Fin


In [3]:
# Initializes array of density values
cp.random.seed(7) # Opens a random number generator instance, seed 7

rho_A = rho_mean * (1.0 + 0.01 * cp.random.uniform(low=-1, high=1, size=(gridpoints, gridpoints, gridpoints)))
rho_A = cp.maximum(rho_A, 1.E-10)
rho_B = rho_mean * (1.0 + 0.01 * cp.random.uniform(low=-1, high=1, size=(gridpoints, gridpoints, gridpoints)))
rho_B = cp.maximum(rho_B, 1.E-10)

initial_mass = cp.sum(rho_A) + cp.sum(rho_B)

def laplacian_3d(function_array):
    """
    Computes the 3D Laplacian of a function, given an array representing that function
    """
    return ( #Uses the inbuilt roll which does allow for periodic boundary conditions
        cp.roll(function_array,  1, axis=0) +
        cp.roll(function_array, -1, axis=0) +
        cp.roll(function_array,  1, axis=1) +
        cp.roll(function_array, -1, axis=1) +
        cp.roll(function_array,  1, axis=2) +
        cp.roll(function_array, -1, axis=2) -
        6.0 * function_array) * inv_dx2



beta_mu_kernel_A = cp.ElementwiseKernel(
    'float64 rho_A, float64 rho_B, float64 lap_rho_A',
    'float64 output',
    f'''
    double Ca = (rho_A + 0.25 * rho_B) * {valence} * {Da};
    double Xa = (-1.0 + sqrt(1.0 + 4.0 * Ca)) / (2.0 * Ca);
    output = 2.0 * {B2aa} * rho_A + 2.0 * {B2ab} * rho_B
           + log(rho_A)
           + {valence} * log(Xa)
           - {Ka} * lap_rho_A;
    ''',
    'beta_mu_kernel_A'
)

beta_mu_kernel_B = cp.ElementwiseKernel(
    'float64 rho_A, float64 rho_B, float64 lap_rho_B',
    'float64 output',
    f'''
    double Ca = (rho_A + 0.25 * rho_B) * {valence} * {Da};
    double Cb = (0.75 * rho_B) * {valence} * {Db};
    double Xa = (-1.0 + sqrt(1.0 + 4.0 * Ca)) / (2.0 * Ca);
    double Xb = (-1.0 + sqrt(1.0 + 4.0 * Cb)) / (2.0 * Cb);
    output = 2.0 * {B2ab} * rho_A + 2.0 * {B2bb} * rho_B
           + log(rho_B)
           + ({valence} / 4.0) * (log(Xa) + 3 * log(Xb))
           - {Kb} * lap_rho_B;
    ''',
    'beta_mu_kernel_B'
)




def compute_step_two(rho_A, rho_B):
    lap_A = laplacian_3d(rho_A)
    lap_B = laplacian_3d(rho_B)

    mu_A = beta_mu_kernel_A(rho_A, rho_B, lap_A)
    mu_B = beta_mu_kernel_B(rho_A, rho_B, lap_B)

    rho_A_step = dt * M * laplacian_3d(mu_A)
    rho_B_step = dt * M * laplacian_3d(mu_B)

    return rho_A_step, rho_B_step
    



def save_density(index, rho_A_total_array, rho_B_total_array, output_dir, sim_params,
                  channel_colors=None):

    rho_A_final = rho_A_total_array[index]
    if hasattr(rho_A_final, "get"):
        rho_A_final = rho_A_final.get()
    rho_A_final = rho_A_final.astype(np.float32)

    rho_B_final = rho_B_total_array[index]
    if hasattr(rho_B_final, "get"):
        rho_B_final = rho_B_final.get()
    rho_B_final = rho_B_final.astype(np.float32)

    if channel_colors is None:
        channel_colors = [(1, 0, 0), (0, 1, 0)]  # red = A, green = B

    zarr_path = os.path.join(output_dir, "final_density.zarr")
    root = zarr.open_group(zarr_path, mode="w")
    root.create_array("component_0", data=rho_A_final, chunks=(64, 64, 64))
    root.create_array("component_1", data=rho_B_final, chunks=(64, 64, 64))

    metadata = {
        "n_components": 2,
        "gridpoints": rho_A_final.shape[0],
        "channels": [
            {
                "name": "component_0",
                "color": channel_colors[0],
                "contrast_limits": [float(rho_A_final.min()), float(rho_A_final.max())],
            },
            {
                "name": "component_1",
                "color": channel_colors[1],
                "contrast_limits": [float(rho_B_final.min()), float(rho_B_final.max())],
            },
        ],
        "sim_params": sim_params,
    }
    with open(os.path.join(output_dir, "render_metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    return zarr_path
    

In [ ]:
# Initializes arrays for saving rho
num_saves = duration // save_interval + 1
rho_A_total_array = cp.zeros((num_saves, gridpoints, gridpoints, gridpoints))
rho_B_total_array = cp.zeros((num_saves, gridpoints, gridpoints, gridpoints))
rho_A_total_array[0] = rho_A
rho_B_total_array[0] = rho_B
save_index = 1


# Tracks the mass over time to ensure conservation
mass_history = []
time_history = []

os.makedirs("OUTPUTS", exist_ok=True)
output_dir = os.path.join("OUTPUTS", f"3D_A_{length_a}_B_{length_b}")
os.makedirs(output_dir, exist_ok=True)
progress_file = os.path.join(output_dir, f"3D_{num_saves-1}_timesteps_progress.txt")


start_time = time.perf_counter()
for step in range(duration):

    # Iterates to find new values of rho_A, rho_B
    rho_A_step, rho_B_step = compute_step_two(rho_A, rho_B)
    rho_A += rho_A_step
    rho_B += rho_B_step

    # Adds the new densities to the arrays + checks mass conservation
    if step % (save_interval) == 0:
        rho_A_total_array[save_index] = rho_A
        rho_B_total_array[save_index] = rho_B

        total_mass = cp.sum(rho_A) + cp.sum(rho_B)

        mass_history.append(total_mass)
        time_history.append(step * dt)

        rho_A = cp.maximum(rho_A, floor)
        rho_B = cp.maximum(rho_B, floor)

        time_elapsed = time.perf_counter() - start_time
        with open(progress_file, "w") as f:
            f.write(f"Progress: {step // save_interval} out of {duration // save_interval} "
                     f"at {time_elapsed:.1f} seconds\n")

        if step % (save_interval * 10) == 0:
            save_density(
                save_index,
                rho_A_total_array,
                rho_B_total_array,
                output_dir,
                sim_params={"B2aa": B2aa, "B2ab": B2ab, "B2bb": B2bb,
                            "valence": valence, "Ka": Ka, "Kb": Kb,
                            "recent_step": step,
                            "max_A": float(rho_A.max()), "min_A": float(rho_A.min()),
                            "max_B": float(rho_B.max()), "min_B": float(rho_B.min())},
            )
        save_index += 1